In [1]:
import pandas as pd
import numpy as np
import os
from datetime import time

In [2]:
!pwd

/home/dizhihuang/graduate/predict_workflow


In [3]:
# 处理时间格式（例如 635 => 06:35）
def convert_numeric_time(val):
    try:
        val = int(val)
        hour = val // 100
        minute = val % 100
        return time(hour=hour, minute=minute)
    except:
        return None

In [4]:

df_all = pd.DataFrame()

# 自定义函数示例（请根据你自己的函数定义实际添加）
def convert_numeric_time(val):
    # 示例处理函数：如果是数字就转换为时间格式字符串（例如：830 -> '08:30:00'）
    if isinstance(val, (int, float)) and not pd.isna(val):
        h = int(val) // 100
        m = int(val) % 100
        return f"{h:02d}:{m:02d}:00"
    return val  # 如果是已经是字符串就直接返回

for file in os.listdir("/home/dizhihuang/graduate/predict_workflow/data/meta_data"):
    if file.endswith(".xlsx"):
        print(f"处理文件：{file}")
        df = pd.read_excel(f"/home/dizhihuang/graduate/predict_workflow/data/meta_data/{file}")
        df_cleaned = df.copy()

        # 转换时间格式
        for col in ['発生時刻', 'ピーク時刻']:
            df_cleaned[col] = df_cleaned[col].apply(convert_numeric_time)

        # 转换并保留一位小数
        for col in ['ピーク長', '発生Ｋｐ', '発生時渋滞長']:
            df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')
            df_cleaned[col] = df_cleaned[col].div(10).round(1)

        # 选取需要的列
        df_clean = df_cleaned[['年', '月', '日', '上下','原因','道路番号', '発生時刻', 'ピーク時刻', 'ピーク長', '発生Ｋｐ', '発生時渋滞長', '渋滞時間']].copy()
        df_jam = df_clean[df_clean['原因'] == '交通集中']
        # 拼接到总表中
        df_all = pd.concat([df_all, df_jam], ignore_index=True)
    


处理文件：★2021_関東支社渋滞データ（01-12）SIC分割 【コード変換・BT記入・本社BT】特定更新工事v3.xlsx
处理文件：★2023_関東支社渋滞データ（01-12）SIC分割【コード変換・BT記入・本社BT】特定更新工事-緊急工事(1～12月分まで).xlsx
处理文件：★2024_関東支社渋滞データ（01-12）SIC分割【コード変換・BT記入・本社BT】特定更新工事-緊急工事(1～12月分まで).xlsx
处理文件：★2025_関東支社渋滞データ（01-03）SIC分割【コード変換・BT記入・本社BT】特定更新工事-緊急工事.xlsx
处理文件：★2022_関東支社渋滞データ（01-12）SIC分割【コード変換・BT記入・本社BT】特定更新工事v3.xlsx
处理文件：★2025_関東支社渋滞データ（01-05）SIC分割【コード変換・BT記入・本社BT】特定更新工事-緊急工事.xlsx
处理文件：★2015_関東支社渋滞データ（01-12）【コード変換・BT記入】.xlsx
处理文件：★2016_関東支社渋滞データ（01-12）SIC分割【コード変換・BT記入】.xlsx
处理文件：★2017_関東支社渋滞データ（01-12）SIC分割【コード変換・BT記入】特定更新工事.xlsx
处理文件：★2018_関東支社渋滞データ（01-12）SIC分割【コード変換・BT記入】特定更新工事.xlsx
处理文件：★2019_関東支社渋滞データ（01-12）SIC分割【コード変換・BT記入・本社BT】v3.xlsx
处理文件：★2020_関東支社渋滞データ（01-12）SIC分割【コード変換・BT記入・本社BT】v3.xlsx
处理文件：★2014_関東支社渋滞データ（01-12）【コード変換・BT記入】最新.xlsx
处理文件：★2025_関東支社渋滞データ（01-11）SIC分割【コード変換・BT記入・本社BT】特定更新工事-緊急工事.xlsx


In [5]:
df_all

,年,月,日,上下,原因,道路番号,発生時刻,ピーク時刻,ピーク長,発生Ｋｐ,発生時渋滞長,渋滞時間
0,2021,1,13,上,交通集中,東北道,07:45:00,07:45:00,1.7,0.6,1.7,70
1,2021,1,14,上,交通集中,東北道,07:35:00,08:02:00,1.7,0.6,1.7,55
2,2021,1,15,上,交通集中,東北道,07:20:00,07:20:00,1.7,0.6,1.7,95
3,2021,1,21,上,交通集中,東北道,08:00:00,08:22:00,1.7,0.6,1.7,45
4,2021,1,25,上,交通集中,東北道,07:10:00,07:25:00,1.7,0.6,1.7,30
...,...,...,...,...,...,...,...,...,...,...,...,...
239920,2025,11,28,外,交通集中,圏央道,09:30:00,10:00:00,7.3,105.7,0.0,75
239921,2025,11,28,外,交通集中,圏央道,14:25:00,14:25:00,3.3,105.7,3.3,25
239922,2025,11,28,外,交通集中,圏央道,16:10:00,17:45:00,7.3,105.7,0.0,190
239923,2025,11,28,外,交通集中,圏央道,06:45:00,06:50:00,3.9,122.0,0.0,105


In [6]:
df_all['date'] = pd.to_datetime(
    df_all['年'].astype(str) + '-' +
    df_all['月'].astype(str).str.zfill(2) + '-' +
    df_all['日'].astype(str).str.zfill(2),
    errors='coerce'
)

df_all.drop(columns=['年', '月', '日'], inplace=True)

# ========== 去重逻辑 ==========
# 基于所有关键列去重，保留第一条记录
before_count = len(df_all)
print(f"去重前记录数: {before_count}")

df_all = df_all.drop_duplicates(
    subset=['date', '上下', '道路番号', '発生時刻', 'ピーク時刻', 'ピーク長', '発生Ｋｐ', '発生時渋滞長', '渋滞時間'],
    keep='first'
)

after_count = len(df_all)
print(f"去重后记录数: {after_count}")
print(f"删除重复记录: {before_count - after_count} 条")

去重前记录数: 239925
去重后记录数: 221868
删除重复记录: 18057 条


In [7]:
df_all

,上下,原因,道路番号,発生時刻,ピーク時刻,ピーク長,発生Ｋｐ,発生時渋滞長,渋滞時間,date
0,上,交通集中,東北道,07:45:00,07:45:00,1.7,0.6,1.7,70,2021-01-13
1,上,交通集中,東北道,07:35:00,08:02:00,1.7,0.6,1.7,55,2021-01-14
2,上,交通集中,東北道,07:20:00,07:20:00,1.7,0.6,1.7,95,2021-01-15
3,上,交通集中,東北道,08:00:00,08:22:00,1.7,0.6,1.7,45,2021-01-21
4,上,交通集中,東北道,07:10:00,07:25:00,1.7,0.6,1.7,30,2021-01-25
...,...,...,...,...,...,...,...,...,...,...
239920,外,交通集中,圏央道,09:30:00,10:00:00,7.3,105.7,0.0,75,2025-11-28
239921,外,交通集中,圏央道,14:25:00,14:25:00,3.3,105.7,3.3,25,2025-11-28
239922,外,交通集中,圏央道,16:10:00,17:45:00,7.3,105.7,0.0,190,2025-11-28
239923,外,交通集中,圏央道,06:45:00,06:50:00,3.9,122.0,0.0,105,2025-11-28


In [8]:
df_all = df_all[['date','上下','原因','道路番号', '発生時刻', 'ピーク時刻', 'ピーク長', '発生Ｋｐ', '発生時渋滞長', '渋滞時間']]
df_all.head()

,date,上下,原因,道路番号,発生時刻,ピーク時刻,ピーク長,発生Ｋｐ,発生時渋滞長,渋滞時間
0,2021-01-13,上,交通集中,東北道,07:45:00,07:45:00,1.7,0.6,1.7,70
1,2021-01-14,上,交通集中,東北道,07:35:00,08:02:00,1.7,0.6,1.7,55
2,2021-01-15,上,交通集中,東北道,07:20:00,07:20:00,1.7,0.6,1.7,95
3,2021-01-21,上,交通集中,東北道,08:00:00,08:22:00,1.7,0.6,1.7,45
4,2021-01-25,上,交通集中,東北道,07:10:00,07:25:00,1.7,0.6,1.7,30


In [9]:
df_all.to_csv("/home/dizhihuang/graduate/predict_workflow/data/processed_data/all_data.csv", index=False)

In [10]:
import os

# 保存路径
output_dir = "/home/dizhihuang/graduate/predict_workflow/data/processed_data"
os.makedirs(output_dir, exist_ok=True)

# 你想保留的道路列表
target_roads = ['東北道', '関越道']

# 只保留目标道路的数据
df_filtered = df_all[df_all['道路番号'].isin(target_roads)]

# 分组后保存
grouped = df_filtered.groupby(['date', '道路番号', '上下'])

for (date, road, direction), group_df in grouped:
    year = date.year
    month_day = date.strftime('%m-%d')
    filename = f"{road}_{direction}_{year}_{month_day}.csv"
    filepath = os.path.join(output_dir, filename)
    
    group_df.to_csv(filepath, index=False)

In [11]:
from datetime import time

df_cleaned = df.copy()

# 处理时间格式（例如 635 => 06:35）
def convert_numeric_time(val):
    try:
        val = int(val)
        hour = val // 100
        minute = val % 100
        return time(hour=hour, minute=minute)
    except:
        return None

for col in ['発生時刻', 'ピーク時刻']:
    df_cleaned[col] = df_cleaned[col].apply(convert_numeric_time)

# 处理保留一位小数的字段（如 12 -> 1.2）
for col in ['ピーク長', '発生Ｋｐ', '発生時渋滞長']:
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')
    df_cleaned[col] = df_cleaned[col].div(10).round(1)

# （可选）保存
# df_cleaned.to_excel("関越2024_cleaned.xlsx", index=False)

print(df_cleaned[['発生時刻', 'ピーク時刻', 'ピーク長', '発生Ｋｐ', '発生時渋滞長']].head())

       発生時刻     ピーク時刻  ピーク長  発生Ｋｐ  発生時渋滞長
0  11:15:00  11:15:00   1.7   0.6     1.7
1  12:20:00  12:20:00   1.7   0.6     1.7
2  13:35:00  14:25:00   4.2  10.5     0.0
3  19:50:00  19:50:00  12.0  10.5    12.0
4  20:45:00  20:45:00   6.4  10.5     6.4


In [12]:
df_cleaned[['発生時刻', 'ピーク時刻', 'ピーク長', '発生Ｋｐ', '発生時渋滞長']].head()


,発生時刻,ピーク時刻,ピーク長,発生Ｋｐ,発生時渋滞長
0,11:15:00,11:15:00,1.7,0.6,1.7
1,12:20:00,12:20:00,1.7,0.6,1.7
2,13:35:00,14:25:00,4.2,10.5,0.0
3,19:50:00,19:50:00,12.0,10.5,12.0
4,20:45:00,20:45:00,6.4,10.5,6.4


In [13]:
df_final = df_cleaned[['年' ,'月' ,'日' ,'発生時刻', 'ピーク時刻', 'ピーク長', '発生Ｋｐ', '発生時渋滞長', '渋滞時間']]
df_final.head()

df_final.to_excel("関越2024_cleaned.xlsx", index=False)

df_final.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37233 entries, 0 to 37232
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   年       37233 non-null  int64  
 1   月       37233 non-null  int64  
 2   日       37233 non-null  int64  
 3   発生時刻    37233 non-null  object 
 4   ピーク時刻   37233 non-null  object 
 5   ピーク長    37233 non-null  float64
 6   発生Ｋｐ    37233 non-null  float64
 7   発生時渋滞長  37233 non-null  float64
 8   渋滞時間    37233 non-null  int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 2.6+ MB
